# A3 — Econometric and residual diagnostics

This notebook reads saved Phase A3 outputs. It does not retrain models.

The residual autocorrelation tests use horizon-spaced residuals so that serial dependence is not mechanically created by overlapping future-return targets. The final 20% exploratory holdout remains unused.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

current = Path.cwd()
project_root = current.parent if current.name == "notebooks" else current
tables = project_root / "reports" / "tables"

summary = pd.read_csv(tables / "diagnostics_summary.csv")
folds = {
    h: pd.read_csv(tables / f"diagnostics_h{h}_folds.csv")
    for h in (10, 50, 100)
}
time_buckets = {
    h: pd.read_csv(tables / f"diagnostics_h{h}_time_buckets.csv")
    for h in (10, 50, 100)
}
summary

## 1. Market return dependence

In [ ]:
display(summary[[
    "horizon",
    "mean_mid_return_acf_lag_1",
    "mean_squared_mid_return_acf_lag_1",
]])

ax = summary.plot(
    x="horizon",
    y=["mean_mid_return_acf_lag_1", "mean_squared_mid_return_acf_lag_1"],
    kind="bar",
    figsize=(9, 5),
)
ax.axhline(0.0, linewidth=1)
ax.set_ylabel("Mean fold autocorrelation")
ax.set_title("Lag-1 dependence in returns and squared returns")
plt.tight_layout()
plt.show()

## 2. Non-overlapping residual dependence

In [ ]:
display(summary[[
    "horizon",
    "mean_residual_acf_lag_1",
    "mean_absolute_residual_acf_lag_1",
    "residual_ljung_box_20_rejections_5pct",
]])

combined = []
for horizon, frame in folds.items():
    selected = frame[["fold", "residual_acf_lag_1"]].copy()
    selected["horizon"] = horizon
    combined.append(selected)
residual_acf = pd.concat(combined, ignore_index=True)

pivot = residual_acf.pivot(index="fold", columns="horizon", values="residual_acf_lag_1")
ax = pivot.plot(marker="o", figsize=(9, 5))
ax.axhline(0.0, linewidth=1)
ax.set_ylabel("Residual ACF at lag 1")
ax.set_title("Non-overlapping Ridge residual autocorrelation")
plt.tight_layout()
plt.show()

## 3. Forecast calibration

In [ ]:
display(summary[[
    "horizon",
    "mean_calibration_slope",
    "mean_calibration_intercept_bps",
    "mean_residual_mean_bps",
    "mean_residual_std_bps",
]])

calibration = []
for horizon, frame in folds.items():
    selected = frame[["fold", "calibration_slope"]].copy()
    selected["horizon"] = horizon
    calibration.append(selected)
calibration = pd.concat(calibration, ignore_index=True)

pivot = calibration.pivot(index="fold", columns="horizon", values="calibration_slope")
ax = pivot.plot(marker="o", figsize=(9, 5))
ax.axhline(1.0, linewidth=1)
ax.set_ylabel("Calibration slope")
ax.set_title("Actual return regressed on predicted return")
plt.tight_layout()
plt.show()

## 4. Out-of-sample performance by clock-time bucket

In [ ]:
for horizon, frame in time_buckets.items():
    print(f"{horizon}-event horizon")
    display(frame[[
        "time_bucket",
        "observations",
        "mean_spread_bps",
        "one_event_return_std_bps",
        "rank_ic",
        "mae_improvement_pct",
        "nonzero_directional_accuracy",
        "residual_mean_bps",
    ]])

In [ ]:
for metric, title in [
    ("rank_ic", "Rank IC by time bucket"),
    ("mae_improvement_pct", "Ridge MAE improvement by time bucket"),
    ("nonzero_directional_accuracy", "Non-zero directional accuracy by time bucket"),
]:
    comparison = None
    for horizon, frame in time_buckets.items():
        series = frame.set_index("time_bucket")[metric].rename(str(horizon))
        comparison = series.to_frame() if comparison is None else comparison.join(series, how="outer")
    ax = comparison.plot(marker="o", figsize=(11, 5))
    if metric == "rank_ic" or metric == "mae_improvement_pct":
        ax.axhline(0.0, linewidth=1)
    if metric == "nonzero_directional_accuracy":
        ax.axhline(0.50, linewidth=1)
    ax.set_title(title)
    ax.set_xlabel("Clock-time bucket")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

## Interpretation checklist

- Negative lag-1 mid-return autocorrelation can indicate short-term quote reversal or bid–ask/microstructure effects.
- Positive squared-return autocorrelation is evidence of volatility clustering.
- Ljung–Box rejection on horizon-spaced residuals means the linear model leaves serial structure unexplained.
- Calibration slope near 1 and intercept near 0 are ideal; a low slope indicates that predicted magnitudes are not well calibrated.
- Time buckets identify where the model becomes biased or loses accuracy. Phase B will formally test spread, liquidity, time-of-day and execution-cost regimes rather than tune the model here.